# FinBERT Item Analysis Runner

Colab wrapper for `finbert_item_analysis_runner.py`. It uses the Drive full-data SEC-CCM output discovered at `/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner` and keeps the script runner as the execution authority.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ENV_VAR = "NLP_THESIS_REPO_ROOT"
GIT_URL_ENV_VAR = "NLP_THESIS_GIT_URL"
GIT_REF_ENV_VAR = "NLP_THESIS_GIT_REF"
DEFAULT_GIT_URL = "https://github.com/Erew42/NLP_Thesis.git"
DEFAULT_GIT_REF = "main"
CLONE_ROOT = Path("/content/NLP_Thesis")


def _find_repo_root() -> Path | None:
    candidates: list[Path] = []
    env_root = os.environ.get(REPO_ENV_VAR)
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    if IN_COLAB:
        candidates.extend(
            [
                CLONE_ROOT,
                Path("/content/drive/MyDrive/NLP_Thesis"),
                Path("/content/drive/My Drive/NLP_Thesis"),
            ]
        )

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "thesis_pkg" / "pipeline.py").exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

repo_root = _find_repo_root()
if repo_root is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / "src" / "thesis_pkg" / "pipeline.py").exists():
        raise FileExistsError(
            f"{CLONE_ROOT} exists but is not the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}."
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_ROOT)])
    repo_root = CLONE_ROOT

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Thesis checkout. Run from the repo root, set NLP_THESIS_REPO_ROOT, or use Colab."
    )

repo_root = repo_root.resolve()
if IN_COLAB and (repo_root / ".git").exists() and os.environ.get("NLP_THESIS_SKIP_GIT_UPDATE", "0") != "1":
    git_ref = os.environ.get(GIT_REF_ENV_VAR, DEFAULT_GIT_REF)
    fetch_code = subprocess.call(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", git_ref])
    checkout_target = "FETCH_HEAD" if fetch_code == 0 else git_ref
    subprocess.call(["git", "-C", str(repo_root), "checkout", checkout_target])

os.environ[REPO_ENV_VAR] = str(repo_root)
src = repo_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB and os.environ.get("NLP_THESIS_SKIP_PIP_INSTALL", "0") != "1":
    install_target = f"{repo_root}[benchmark]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--editable", install_target])

print({"IN_COLAB": IN_COLAB, "repo_root": str(repo_root), "src_exists": src.exists()})

In [ ]:
from pathlib import Path


def _resolve_colab_drive_root() -> Path:
    for candidate in (
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive"),
    ):
        if candidate.exists():
            return candidate
    return Path("/content/drive")


def _first_existing_path(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def _check_paths(paths: dict[str, Path], *, require_all: bool = True) -> None:
    rows = []
    missing = []
    for label, path in paths.items():
        exists = path.exists()
        rows.append({"label": label, "path": str(path), "exists": exists})
        if require_all and not exists:
            missing.append(f"{label}: {path}")
    try:
        import polars as pl

        display(pl.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)
    if missing:
        raise FileNotFoundError("Missing required Drive inputs:\n" + "\n".join(missing))

DRIVE_ROOT = _resolve_colab_drive_root()
WORK_ROOT = DRIVE_ROOT / "Data_LM"
SEC_CCM_RUN_ROOT = WORK_ROOT / "results" / "sec_ccm_unified_runner"

SOURCE_ITEMS_DIR = SEC_CCM_RUN_ROOT / "items_analysis"
BACKBONE_PATH = SEC_CCM_RUN_ROOT / "sec_ccm_premerge" / "final_flagged_data.parquet"
OUTPUT_DIR = WORK_ROOT / "results" / "benchmarking" / "finbert_item_analysis_runner"

RUN_NAME = "finbert_item_analysis_full_data"
YEAR_FILTER = None  # Example smoke run: [2006, 2007, 2008]
BATCH_PROFILE = "baseline"  # small | baseline | large
DEVICE = None  # None lets torch choose; use "cuda" to require GPU.
PREPROCESS_ONLY = False
OVERWRITE = False
WRITE_SENTENCE_SCORES = False
NOTE = "Colab run using Drive full-data sec_ccm_unified_runner outputs."

_check_paths(
    {
        "WORK_ROOT": WORK_ROOT,
        "SOURCE_ITEMS_DIR": SOURCE_ITEMS_DIR,
        "BACKBONE_PATH": BACKBONE_PATH,
    }
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"OUTPUT_DIR": str(OUTPUT_DIR), "RUN_NAME": RUN_NAME})

In [ ]:
from thesis_pkg.notebooks_and_scripts import finbert_item_analysis_runner as runner

RUN_ARGS = [
    "--data-profile", "EXPLICIT",
    "--source-items-dir", str(SOURCE_ITEMS_DIR),
    "--backbone-path", str(BACKBONE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--batch-profile", BATCH_PROFILE,
    "--note", NOTE,
]
if RUN_NAME:
    RUN_ARGS.extend(["--run-name", RUN_NAME])
if YEAR_FILTER is not None:
    RUN_ARGS.extend(["--years", *(str(year) for year in YEAR_FILTER)])
if DEVICE is not None:
    RUN_ARGS.extend(["--device", DEVICE])
if PREPROCESS_ONLY:
    RUN_ARGS.append("--preprocess-only")
if OVERWRITE:
    RUN_ARGS.append("--overwrite")
if WRITE_SENTENCE_SCORES:
    RUN_ARGS.append("--write-sentence-scores")

RUN_ARGS

In [ ]:
exit_code = runner.main(RUN_ARGS)
assert exit_code == 0

In [ ]:
import json
import polars as pl

run_dir = OUTPUT_DIR / RUN_NAME if RUN_NAME else max((p for p in OUTPUT_DIR.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
manifest_path = run_dir / "run_manifest.json"
print({"run_dir": str(run_dir), "manifest_path": str(manifest_path)})

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps({"counts": manifest.get("counts"), "warnings": manifest.get("warnings")}, indent=2))

for candidate in (run_dir / "item_features_long.parquet", run_dir / "doc_features_wide.parquet", run_dir / "coverage_report.parquet"):
    if candidate.exists():
        print(candidate.name, pl.scan_parquet(candidate).select(pl.len().alias("rows")).collect().item())